# 3-2 MCP: Model Context Protocol

**원본 교안:** [3-2-model-context-protocol.pdf](./3-2-model-context-protocol.pdf) · PDF는 그대로 보존합니다.

이 노트북은 슬라이드의 개념 순서를 따라 설명과 확인 실습을 덧붙인 학습본입니다. 로컬 예제는 표준 라이브러리만 사용합니다. 실제 사내 도구·서버 연결은 강의장 환경에서 별도로 확인하세요.


## 학습 목표

- Function Calling 연결 코드가 늘 때 생기는 문제를 설명한다.
- Server·Host·Client와 Tool·Resource·Prompt의 역할을 구분한다.
- 슬라이드의 SSE 설정과 실제 실습 파일의 연결 방식을 대조한다.


> **실습 전 확인**  사내망과 외부망에서 서버 실행 가능 여부가 다를 수 있습니다. 슬라이드의 SSE 설정을 실행 검증 없이 연결 완료라고 판단하지 마세요.


## 학습 경로 · PDF 1–3쪽

[원본 1~3쪽](./3-2-model-context-protocol.pdf#page=1)

함수 호출의 한계 → MCP 구성요소 → 서버 제공 기능 → 클라이언트 연결 순서입니다.


## 함수 호출과 MCP · PDF 4–5쪽

[원본 4~5쪽](./3-2-model-context-protocol.pdf#page=4)

도구 이름·설명·입력 스키마를 앱마다 직접 관리하면 재사용과 권한 관리가 어려워집니다. MCP는 도구와 데이터를 Server에서 제공하고, Host의 Client가 이를 가져오는 표준 연결 방식을 제시합니다. 프로토콜이 있다고 해서 도구 권한이 자동으로 안전해지는 것은 아닙니다.


## 세 구성요소 · PDF 6쪽

[원본 6쪽](./3-2-model-context-protocol.pdf#page=6)

**Server**는 기능과 데이터를 제공하고, **Host**는 사용자가 쓰는 LLM 앱이며, **Client**는 Host 안에서 Server와 통신합니다. 하나의 Host가 여러 Server에 연결할 수 있습니다. 6쪽 도식에서는 Host 안의 여러 MCP Client가 로컬 또는 원격 Server로 각각 연결되는 모양을 보여 줍니다.


## FastMCP와 세 Primitive · PDF 7–9쪽

[원본 7~9쪽](./3-2-model-context-protocol.pdf#page=7)

슬라이드는 FastMCP의 데코레이터로 Tool(행동), Resource(읽을 자료), Prompt(재사용 지시)를 정의합니다. 8쪽 서버 예제는 `transport="sse"`, 포트 8000으로 실행하며, 9쪽에는 BMI Tool·앱 정보 Resource·작업 요약 Prompt 예시가 있습니다. 어떤 기능을 Tool로 공개할지 선택하고 입력을 검증해야 합니다.


## LangChain 클라이언트 · PDF 10–12쪽

[원본 10~12쪽](./3-2-model-context-protocol.pdf#page=10)

슬라이드는 `MultiServerMCPClient` 설정에 `transport="sse"`와 `http://localhost:8000/sse`를 넣고 `await client.get_tools()`로 Tool을 가져온 뒤 모델에 연결합니다. 현재 `Chapter02/01-mcp/` 실습의 전송 방식은 슬라이드와 다를 수 있으므로 실행 전에 서버·클라이언트 양쪽 설정이 일치하는지 확인하세요.


### 실행해 보기: Tool 입력 스키마 확인

실제 네트워크 연결 전, 함수에 필요한 인수를 명확히 적는 연습입니다.


In [ ]:
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    if weight_kg <= 0 or height_cm <= 0:
        raise ValueError("몸무게와 키는 양수여야 합니다")
    bmi = weight_kg / (height_cm / 100) ** 2
    return {"bmi": round(bmi, 1)}

print(calculate_bmi(70, 175))


## CodeMate와 페어 프로그래밍

아래 문장을 CodeMate에 전달하고, 생성된 파일과 실행 결과를 직접 검토하세요. 사내 확장의 메뉴 이름이나 동작은 현장에서 확인합니다.

> Chapter02/01-mcp의 서버와 클라이언트 파일을 읽고 전송 방식(SSE 또는 stdio), 주소·명령, 공개 Tool을 표로 정리해 줘. 코드를 바꾸기 전에 양쪽이 실제로 연결되는지 검증할 최소 실행 절차와 실패 시 확인할 로그를 제안해 줘.


## 결과 검토와 되돌아보기

1. BMI 함수를 Tool로 노출할 때 유효하지 않은 입력은 어떻게 처리해야 하는가?
2. Resource와 Tool 중 파일 내용 읽기를 어느 쪽으로 제공할지 어떤 기준으로 결정하는가?
3. 슬라이드와 로컬 코드의 전송 방식이 다르면 어떤 증상이 나는가?
